
# Project : Multimodal RAG (Charts, Tables and Text Pages)
**Domain:** Financial / Business Document Q&A | **Level:** Deep-Tech | **Stack:** Python, PyMuPDF, Groq vision model, Sentence-Transformers, Matplotlib

## Problem statement
A lot of useful business information sits inside charts and tables. Normal text extraction from a PDF loses it or turns it into messy text. In this project I turn each PDF page into an image, ask a vision model to write a detailed description of it, index those descriptions for search, and when a question comes I find the right page and show the original page image to the vision model to get the answer.

## What I built
- A 4 page business report PDF made in the notebook (bar chart, line chart, table image, text page)
- PyMuPDF page renderer and plain text extraction (to show what normal extraction misses)
- Vision model summaries for every page, stored as searchable vectors
- Retrieval that returns the original page image (not just the summary)
- Evaluation: right page retrieved? right answer given?

**Vision model note:** Groq changes its vision models often (Llama 4 Scout was retired in June 2026). This notebook tries the models listed in `VISION_MODELS`. If both fail, check https://console.groq.com/docs/vision and put a current vision model name in that list.

## Step 0: Install libraries

In [1]:
!pip install -q pymupdf sentence-transformers openai pandas matplotlib faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.8 MB/s eta 0:00:00


## Step 1: Connect to Groq

In [2]:
import os, json, time, re
from getpass import getpass
from openai import OpenAI

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key here: ")

client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
MODEL = "openai/gpt-oss-120b"

def call_llm(prompt, system="You are a helpful assistant.", json_mode=False):
    for attempt in range(3):
        try:
            msgs = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
            if json_mode:
                res = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0, response_format={"type": "json_object"})
            else:
                res = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0)
            return (res.choices[0].message.content or "").strip()
        except Exception as e:
            print("LLM error (try", attempt + 1, "):", str(e)[:150])
            time.sleep(6)
    return ""

print(call_llm("Say hello in 5 words"))

Paste your Groq API key here: ··········
Hello there, nice to meet!


## Step 2: Create the report PDF
I make my own PDF with charts so there is no download needed, and I know the right answers to test against.
- Page 1: bar chart of quarterly revenue by region
- Page 2: line chart of monthly active users
- Page 3: product table
- Page 4: a text update about churn

In [3]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("company_report.pdf") as pdf:
    # page 1 bar chart
    fig = plt.figure(figsize=(8, 5.5))
    quarters = ["Q1", "Q2", "Q3", "Q4"]
    north = [120, 135, 150, 170]
    south = [90, 95, 110, 105]
    east = [70, 85, 80, 120]
    west = [100, 110, 125, 140]
    x = range(4)
    plt.bar([i - 0.3 for i in x], north, width=0.2, label="North")
    plt.bar([i - 0.1 for i in x], south, width=0.2, label="South")
    plt.bar([i + 0.1 for i in x], east, width=0.2, label="East")
    plt.bar([i + 0.3 for i in x], west, width=0.2, label="West")
    plt.xticks(list(x), quarters)
    plt.title("Quarterly Revenue by Region (in Rs lakh) - 2025")
    plt.ylabel("Revenue (Rs lakh)")
    plt.legend()
    pdf.savefig(fig)
    plt.close()

    # page 2 line chart
    fig = plt.figure(figsize=(8, 5.5))
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    users = [10, 12, 15, 17, 21, 24, 28, 30, 35, 41, 46, 52]
    plt.plot(months, users, marker="o")
    for i in range(12):
        plt.text(i, users[i] + 1, str(users[i]), ha="center")
    plt.title("Monthly Active Users (in thousands) - 2025")
    plt.ylabel("Users (thousands)")
    pdf.savefig(fig)
    plt.close()

    # page 3 table
    fig = plt.figure(figsize=(8, 5.5))
    ax = fig.add_subplot(111)
    ax.axis("off")
    table_data = [["Alpha", "1200", "25", "32%"], ["Beta", "800", "40", "41%"], ["Gamma", "450", "90", "55%"], ["Delta", "300", "120", "28%"]]
    tbl = ax.table(cellText=table_data, colLabels=["Product", "Units Sold", "Price (Rs)", "Profit Margin"], loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(13)
    tbl.scale(1, 2)
    ax.set_title("Product Performance Summary", fontsize=15)
    pdf.savefig(fig)
    plt.close()

    # page 4 text
    fig = plt.figure(figsize=(8, 5.5))
    fig.text(0.1, 0.85, "Customer Success Update", fontsize=18)
    fig.text(0.1, 0.65, "Customer churn dropped from 8.5% to 5.1% after the new", fontsize=12)
    fig.text(0.1, 0.60, "onboarding flow was launched in Q3.", fontsize=12)
    fig.text(0.1, 0.48, "Support tickets per 100 users fell from 19 to 14.", fontsize=12)
    fig.text(0.1, 0.36, "Customer satisfaction score is now 4.4 out of 5.", fontsize=12)
    pdf.savefig(fig)
    plt.close()

print("PDF created: company_report.pdf")

PDF created: company_report.pdf


## Step 3: Read the PDF with PyMuPDF
First I check what plain text extraction gives for each page. Then I render every page to a small PNG image (small because Groq has a size limit for images).

In [4]:
import fitz

pdf_doc = fitz.open("company_report.pdf")
page_images = []
page_names = ["Revenue bar chart", "Active users line chart", "Product table", "Customer success text"]

for i in range(len(pdf_doc)):
    page = pdf_doc[i]
    plain_text = page.get_text().strip()
    pix = page.get_pixmap(dpi=80)
    png_bytes = pix.tobytes("png")
    page_images.append(png_bytes)
    print("page", i + 1, "|", page_names[i], "| plain text characters:", len(plain_text), "| image size KB:", round(len(png_bytes) / 1024))

print("\nplain text from page 1 was:", repr(pdf_doc[0].get_text().strip()[:200]))

page 1 | Revenue bar chart | plain text characters: 129 | image size KB: 17
page 2 | Active users line chart | plain text characters: 159 | image size KB: 22
page 3 | Product table | plain text characters: 140 | image size KB: 18
page 4 | Customer success text | plain text characters: 213 | image size KB: 19

plain text from page 1 was: 'Q1\nQ2\nQ3\nQ4\n0\n20\n40\n60\n80\n100\n120\n140\n160\nRevenue (Rs lakh)\nQuarterly Revenue by Region (in Rs lakh) - 2025\nNorth\nSouth\nEast\nWest'


## Step 4: Vision model reads each page
The vision model writes a detailed description of every page including all numbers. These descriptions are what I will search on.

In [5]:
import base64

VISION_MODELS = ["qwen/qwen3.8-27b"]  # qwen3.6-27b was retired by Groq, only qwen3.8-27b is live now

def ask_vision(png_bytes, question):
    b64 = base64.b64encode(png_bytes).decode("utf-8")
    for vm in VISION_MODELS:
        try:
            res = client.chat.completions.create(
                model=vm,
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": "data:image/png;base64," + b64}}]}],
                temperature=0,
                max_tokens=2000,
            )
            text = res.choices[0].message.content or ""
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
            if text:
                return text
        except Exception as e:
            print("vision model", vm, "failed:", str(e)[:150])
    return ""

summaries = []
for i in range(len(page_images)):
    s = ask_vision(page_images[i], "Describe this page in detail for a search index. Include the title, type of chart or table, every label and every number you can read, and the main trend or takeaway.")
    summaries.append(s)
    print("Page", i + 1, "summary (first 250 chars):")
    print(s[:250])
    print()
    time.sleep(2)

Page 1 summary (first 250 chars):
Based on the provided images, here is a detailed description of the page for a search index.

**Title:** Quarterly Revenue by Region (in Rs lakh) - 2025

**Chart Type:** Grouped Bar Chart

**Axes and Labels:**
- **Y-Axis:** Labeled "Revenue (Rs lakh)

Page 2 summary (first 250 chars):
Based on the provided image, here is a detailed description for a search index:

**Title:** Monthly Active Users (in thousands) - 2025

**Chart Type:** Line chart with data points.

**Axes:**
*   **Y-axis (Vertical):** Labeled "Users (thousands)". Th

Page 3 summary (first 250 chars):
Based on the image provided, here is a detailed description for a search index:

**Title:** Product Performance Summary

**Type of Data:** Table

**Column Headers (Labels):**
*   Product
*   Units Sold
*   Price (Rs)
*   Profit Margin

**Data Rows (L

Page 4 summary (first 250 chars):
Based on the visual evidence, here is a detailed description of the page for a search index.

**Page Title:*

## Step 5: Index the summaries
Only the text summaries go into the vector index. The original page images are stored separately in a dictionary (docstore) and found by page number. This is the multi-vector idea: search on text, answer from the original image.

In [6]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

docstore = {}
for i in range(len(page_images)):
    docstore[i] = page_images[i]

summary_vecs = np.array(embed_model.encode(summaries, normalize_embeddings=True), dtype="float32")
index = faiss.IndexFlatIP(summary_vecs.shape[1])
index.add(summary_vecs)

print("pages in index:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

pages in index: 4


## Step 6: Ask questions
1. Embed the question and find the best matching page summary
2. Take the original page image from the docstore
3. Send the image + question to the vision model for the final answer

In [7]:
def ask_report(question):
    qv = np.array(embed_model.encode([question], normalize_embeddings=True), dtype="float32")
    scores, ids = index.search(qv, 1)
    page_id = int(ids[0][0])
    answer = ask_vision(docstore[page_id], "Look at this page and answer the question using only what you can see. Be short and give the exact number.\n\nQuestion: " + question)
    return page_id, answer

page_id, answer = ask_report("Which region had the highest revenue in Q4?")
print("retrieved page:", page_id + 1, "(", page_names[page_id], ")")
print("answer:", answer)

retrieved page: 1 ( Revenue bar chart )
answer: North


## Step 7: Evaluation
5 questions where I know the right page and the right answer. I check two things: was the right page retrieved, and does the answer contain the right value.

In [8]:
import pandas as pd

test_set = [
    ("Which region had the highest revenue in Q4?", 0, "North"),
    ("What was the Q4 revenue of the West region?", 0, "140"),
    ("How many monthly active users were there in December?", 1, "52"),
    ("Which product has the highest profit margin?", 2, "Gamma"),
    ("What happened to customer churn after the new onboarding flow?", 3, "5.1"),
]

rows = []
page_correct = 0
answer_correct = 0

for question, right_page, keyword in test_set:
    got_page, ans = ask_report(question)
    page_ok = got_page == right_page
    ans_ok = keyword.lower() in ans.lower()
    if page_ok:
        page_correct = page_correct + 1
    if ans_ok:
        answer_correct = answer_correct + 1
    rows.append({"question": question, "retrieved_page": got_page + 1, "page_ok": page_ok, "expected": keyword, "answer_ok": ans_ok, "answer": ans[:80]})
    time.sleep(2)

eval_df = pd.DataFrame(rows)
print(eval_df[["question", "retrieved_page", "page_ok", "expected", "answer_ok"]])
print("\nRetrieval accuracy:", page_correct, "/", len(test_set))
print("Answer accuracy:", answer_correct, "/", len(test_set))

                                            question  retrieved_page  page_ok  \
0        Which region had the highest revenue in Q4?               1     True   
1        What was the Q4 revenue of the West region?               1     True   
2  How many monthly active users were there in De...               2     True   
3       Which product has the highest profit margin?               3     True   
4  What happened to customer churn after the new ...               4     True   

  expected  answer_ok  
0    North       True  
1      140       True  
2       52       True  
3    Gamma       True  
4      5.1       True  

Retrieval accuracy: 5 / 5
Answer accuracy: 5 / 5


## Step 8: Results summary

In [9]:
print("Pages indexed:", len(page_images), "(bar chart, line chart, table, text)")
print("Retrieval accuracy:", page_correct, "/", len(test_set))
print("Answer accuracy:", answer_correct, "/", len(test_set))
print()
print("Resume bullet (from my own run):")
print("Built a multimodal RAG pipeline with PyMuPDF, a Groq vision model and FAISS using multi-vector indexing (search on vision-generated summaries, answer from the original page image); on a 4 page report with charts, a table and text it retrieved the right page for", page_correct, "of", len(test_set), "questions and answered", answer_correct, "of", len(test_set), "correctly.")

Pages indexed: 4 (bar chart, line chart, table, text)
Retrieval accuracy: 5 / 5
Answer accuracy: 5 / 5

Resume bullet (from my own run):
Built a multimodal RAG pipeline with PyMuPDF, a Groq vision model and FAISS using multi-vector indexing (search on vision-generated summaries, answer from the original page image); on a 4 page report with charts, a table and text it retrieved the right page for 5 of 5 questions and answered 5 of 5 correctly.


## Key learnings
- Plain PDF text extraction returned almost nothing for chart pages, so vision is really needed
- Searching on summaries but answering from the original image keeps the exact numbers
- Groq limits image size, so I render pages at low dpi to stay under the limit
- Vision model names get retired, so keep them in one list at the top and add a fallback
- My test set is small (5 questions), so real projects need many more questions to trust the numbers

## Skills shown
Multimodal RAG, Vision LLMs, PyMuPDF, Multi-vector retrieval, FAISS, Sentence-Transformers, Matplotlib, Retrieval evaluation, Groq API

## Ideas to extend
- Use a real annual report PDF and split each page into text, table and image parts
- Try ColPali style direct page embeddings and compare
- Add page number citations to every answer

---
**Built by Akshat Kesharwani** | Portfolio: https://akshatkesharwani-info.github.io/ | GitHub: https://github.com/akshatkesharwani-info

Project idea taken from the Dipanshu.Decoded Agentic AI handbook. I rewrote the code to run free on Groq and added my own test questions and evaluation.